# Agentic RAG — Pipeline Demo

## Setup

In [1]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import main as main_module
importlib.reload(main_module)

from main import agentic_rag_app
print("✅ Pipeline loaded successfully.")

c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7997.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Pipeline loaded successfully.


## Helper — Stream Runner

In [2]:
def run_demo(query: str, namespace: str = "default_namespace"):
    """Stream the agentic RAG graph and print a clean per-node trace."""
    initial_state = {
        "query": query,
        "original_query": query,
        "namespace": namespace,
        "documents": [],
        "final_context_strips": [],
        "needs_web_search": False,
        "answer": "",
        "retries": 0,
        "hallucination_retries": 0,
        "route": ""
    }

    print(f"Query : {query}")
    print("=" * 65)

    nodes_visited = []

    for chunk in agentic_rag_app.stream(initial_state):
        for node, update in chunk.items():
            nodes_visited.append(node)
            print(f"\n---> Node [{node}]")

            if "route" in update and update["route"]:
                print(f"  Route              : {update['route']}")
            if "documents" in update:
                print(f"  Docs fetched       : {len(update['documents'])}")
            if "final_context_strips" in update:
                strips = update["final_context_strips"]
                print(f"  Context strips     : {len(strips)}")
            if "needs_web_search" in update:
                print(f"  Needs rewrite/web  : {update['needs_web_search']}")
            if "retries" in update:
                print(f"  Retries            : {update['retries']}")
            if "hallucination_retries" in update:
                print(f"  Halluc retries     : {update['hallucination_retries']}")
            if "answer" in update and update["answer"]:
                print(f"\n  Answer:\n  {update['answer']}")

    print("\n" + "=" * 65)
    print(f"  Node path: {' → '.join(nodes_visited)}")
    print("=" * 65)

---
## LLM Direct

**When triggered:** The Adaptive Router classifies a query as general knowledge that the LLM can answer from its parametric memory — no retrieval needed.

In [3]:
# General-knowledge question — should be classified as llm_direct
run_demo("Explain the concept of entropy in thermodynamics in simple terms.")

Query : Explain the concept of entropy in thermodynamics in simple terms.
2026-03-29 01:10:05 - src.adaptive_router - INFO - Query 'Explain the concept of entropy in thermodynamics in simple terms.' classified to route: 'llm_direct'

---> Node [router]
  Route              : llm_direct

---> Node [generate]

  Answer:
  **Entropy in thermodynamics – a quick, everyday‑language explanation**

1. **Think of “messiness” or “disorder.”**  
   - Imagine a room that’s perfectly tidy: every book is on its shelf, every sock is paired.  
   - Now picture the same room after a party: books are scattered, socks are everywhere.  
   - The messy room has more ways to arrange the same objects – it’s *more disordered*.  
   - In physics, that “messiness” is what we call **entropy**.

2. **How it’s measured**  
   - Entropy is a property of a system (like a gas, a liquid, or a solid).  
   - The more ways the system’s microscopic parts (atoms, molecules) can be arranged while still looking the same on 

In [4]:
# Another example — open-ended reasoning, no document context needed
run_demo("What are the main differences between supervised and unsupervised machine learning?")

Query : What are the main differences between supervised and unsupervised machine learning?
2026-03-29 01:10:12 - src.adaptive_router - INFO - Query 'What are the main differences between supervised and unsupervised machine learning?' classified to route: 'llm_direct'

---> Node [router]
  Route              : llm_direct

---> Node [generate]

  Answer:
  **Supervised vs. Unsupervised Machine Learning**

| Aspect | Supervised Learning | Unsupervised Learning |
|--------|---------------------|-----------------------|
| **Data** | Requires *labeled* data: each input \(x\) comes with a target output \(y\) (e.g., “cat” vs. “dog”). | Uses *unlabeled* data: only the inputs \(x\) are available; no explicit target values. |
| **Goal** | Learn a mapping \(f: X \rightarrow Y\) that can predict the label for new, unseen data. | Discover hidden structure or patterns in the data (e.g., groupings, relationships, low‑dimensional representations). |
| **Typical Tasks** | • Classification (categorical 

---
## Web Search (DuckDuckGo)

**When triggered:** Real-time, current-events, or live-data queries that require information beyond the LLM's training cutoff.

**Node path:** `router → web_search → generate`

**Key behaviours:**
- DuckDuckGo search result is appended as a single context strip with prefix `[Web Search Results]:`
- `hallucination_grader` is **bypassed** — web results are treated as ground truth (`route != 'vectorstore'` short-circuits the check)
- `answer_quality_grader` **does** run — if the answer is off-topic, `rewrite_query` fires and routes back to `web_search` (not vectorstore)
- Hard retry cap: `retries >= 3` → `END` unconditionally

In [5]:
# Real-time query — should be classified as web_search
run_demo("What is the current stock price of NVIDIA (NVDA) and the latest news?")

Query : What is the current stock price of NVIDIA (NVDA) and the latest news?
2026-03-29 01:10:22 - src.adaptive_router - INFO - Query 'What is the current stock price of NVIDIA (NVDA) and the latest news?' classified to route: 'web_search'

---> Node [router]
  Route              : web_search

---> Node [web_search]
  Context strips     : 1
2026-03-29 01:10:24 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  **Current NVIDIA (NVDA) stock price (as of the latest data in the search results)**  
- **$167.52 USD** (down 3.72 points, –2.17 % from the previous close).

**Latest news highlights**  
- **Strong‑Buy rating**: Analysts have upgraded NVDA to a “Strong Buy” and set a 12‑month target price of **$268.22**.  
- **Upside potential**: This target represents roughly a **60.1 % upside** from the current trading level of $167.52.  
- **Market movement**: The share price is trading slightly lower than its previous close, reflecting a modest 2.2

In [6]:
# Another real-time example — current events
run_demo("What are the latest developments in AI regulation in the EU in 2025?")

Query : What are the latest developments in AI regulation in the EU in 2025?
2026-03-29 01:10:28 - src.adaptive_router - INFO - Query 'What are the latest developments in AI regulation in the EU in 2025?' classified to route: 'web_search'

---> Node [router]
  Route              : web_search

---> Node [web_search]
  Context strips     : 1
2026-03-29 01:10:31 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  **Key EU AI‑regulation milestones that unfolded in 2025**

| Date | What happened | Why it matters |
|------|---------------|----------------|
| **February 2025** | **Prohibited AI practices were formally banned** under the EU AI Act. | This was the first concrete enforcement step – any AI system that falls into the prohibited categories (e.g., subliminal manipulation, social scoring, real‑time biometric identification in public spaces) can no longer be deployed. |
| **August 2025** | **General‑purpose AI (GPAI) model obligations kicked 

---
## 📚 Vectorstore

**When triggered:** Document-specific queries where the answer is expected to live in the indexed knowledge base (Pinecone + MongoDB).

**Node path (happy path that we want):** `router → retrieve → evaluate → generate`

**Node path (zero helpful strips):** `router → retrieve → evaluate → rewrite_query → retrieve → evaluate → generate`

**Key behaviours:**

| Stage | What happens |
|---|---|
| **Retrieve** | VectorDB search (+ optional Pinecone Reranker). `parent_id` → MongoDB fetch for full parent chunks |
| **CRAG Evaluate** | Each doc rated `correct / ambiguous / incorrect`. Strips extracted from `correct` and `ambiguous` docs |
| **Zero-strip guard** | `needs_web_search = True` only if `len(final_context_strips) == 0` after ALL docs graded |
| **Self-RAG: Hallucination** | LLM checks answer is grounded in retrieved strips (`yes` / `no`) |
| **Self-RAG: Quality** | LLM checks answer resolves the original query (`yes` / `no`) |
| **Retry hard cap** | `retries >= 3` or `hallucination_retries >= 3` → `END` immediately |

In [7]:
# Document-specific query about the indexed knowledge base (RAG-Anything paper)
run_demo("How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.")

Query : How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.
2026-03-29 01:10:50 - src.adaptive_router - INFO - Query 'How does the hybrid retrieval engine work in RAG-Anything? Explain the two pathways.' classified to route: 'vectorstore'

---> Node [router]
  Route              : vectorstore
2026-03-29 01:10:50 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-29 01:10:52 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)

---> Node [retrieve]
  Docs fetched       : 5
2026-03-29 01:11:34 - src.workflows.crag - INFO - CRAG evaluate: 3 relevant strips found. needs_web_search=False

---> Node [evaluate]
  Context strips     : 3
  Needs rewrite/web  : False
2026-03-29 01:11:48 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-29 01:11:57 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  **Hybrid Retrieval Engine in RAG‑Anything**

RAG‑Anything buil

In [8]:
# Another vectorstore query
run_demo("What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?")

Query : What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?
2026-03-29 01:12:15 - src.adaptive_router - INFO - Query 'What is the dual-graph architecture in RAG-Anything and how do CM-KG and T-KG relate?' classified to route: 'vectorstore'

---> Node [router]
  Route              : vectorstore
2026-03-29 01:12:15 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-29 01:12:17 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)

---> Node [retrieve]
  Docs fetched       : 5
2026-03-29 01:14:09 - src.workflows.crag - INFO - CRAG evaluate: 4 relevant strips found. needs_web_search=False

---> Node [evaluate]
  Context strips     : 4
  Needs rewrite/web  : False
2026-03-29 01:14:22 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-29 01:14:29 - src.workflows.self_rag - INFO - Answer Quality Score: yes

---> Node [generate]

  Answer:
  **Dual‑Graph Architecture in RAG‑Anything**

RAG‑Anything bu